## Predección de precios con modelo finetuneado vs modelo Base

In [ ]:
#instalaciones
!pip install -q datasets==2.21.0
!pip install -q transformers==4.43.1 trl==0.9.6 peft==0.12.0 accelerate==0.32.1
!pip install -q --upgrade bitsandbytes
!pip install -q triton==3.1.0
!pip install numpy==1.26.4 --force-reinstall


In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel, LoraConfig
import matplotlib.pyplot as plt

In [ ]:
# Datos del proyecto, data y usuario de HF
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
PROJECT_NAME = "pricer"
HF_USER = "PamAmezcua"
RUN_NAME = "2026-06-11_13.31.28"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
REVISION = "34734cf60ae758dbfacc5924f7d24e98ed72a06e" # or REVISION = None (para acceder a la última revisión), especifica el commit que deseamos usar. Segun se vaya so reajustando podríamos irnos a un commit anterior, no al último.
FINETUNED_MODEL = f"{HF_USER}/{PROJECT_RUN_NAME}"
DATASET_NAME = f"{HF_USER}/lite-data"

# Hiperparámetros de cuantización del modelo base.
QUANT_4_BIT = True

%matplotlib inline

# Colores usados para marcar en la gráfica aquellas predicciones que hacen "Hit".
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red":RED, "orange": YELLOW, "green": GREEN}

In [ ]:
# Log in en HuggingFace
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
#Descarga de conjunto de train y test de Hugging Face, en este código el conjunto de train no se usa
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

In [ ]:
#Ejemplo
test[0]

In [ ]:
len(train), len(test)

# Funciones adicionales: extracción/predicción de precio y métricas de testeo

In [ ]:
#Función de extracción de precio de un item del conjunto
def extract_price(s):
    if "Price is $" in s:
      contents = s.split("Price is $")[1]
      contents = contents.replace(',','')
      match = re.search(r"[-+]?\d*\.\d+|\d+", contents)
      return float(match.group()) if match else 0
    return 0

In [ ]:
#Ejemplo
extract_price("Price is $99.99")

In [ ]:
#Función de prediccion de precio con finetuned_model_predict
def finetuned_model_predict(prompt):
    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda") #codificar a tokens numericos y mandar a cuda
    attention_mask = torch.ones(inputs.shape, device="cuda") #pequeña linea para evitarnos una advertencia
    outputs = fine_tuned_model.generate(inputs, attention_mask=attention_mask, max_new_tokens=3, num_return_sequences=1) #se le pasa el input codificado, genera maximo de 3 tokens de salida, pero con uno basta (el precio mismo)
    response = tokenizer.decode(outputs[0])#tokenizer decodifica par leerlo en letras.
    return extract_price(response) #de nuestra respuesta se extrae el precio

In [ ]:
#Función de prediccion de precio con base_model
def base_model_predict(prompt):
    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda") #codificar a tokens numericos y mandar a cuda
    attention_mask = torch.ones(inputs.shape, device="cuda") #pequeña linea para evitarnos una advertencia
    outputs = base_model.generate(inputs, attention_mask=attention_mask, max_new_tokens=3, num_return_sequences=1) #se le pasa el input codificado, genera maximo de 3 tokens de salida, pero con uno basta (el precio mismo)
    response = tokenizer.decode(outputs[0])#tokenizer decodifica par leerlo en letras.
    return extract_price(response) #de nuestra respuesta se extrae el precio

In [ ]:
class Tester:

    def __init__(self, predictor, data, title=None, size=250):
        self.predictor = predictor
        self.data = data
        self.title = title or predictor.__name__.replace("_", " ").title()
        self.size = size
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        if  error/truth < 0.2: #error<40 or
            return "green"
        elif error/truth < 0.4: #error<80 or
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        guess = self.predictor(datapoint["text"])
        truth = datapoint["price"]
        error = abs(guess - truth)
        log_error = math.log(truth+1) - math.log(guess+1)
        sle = log_error ** 2
        color = self.color_for(error, truth)
        title = datapoint["text"].split("\n\n")[1][:20] + "..."
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        print(f"{COLOR_MAP[color]}{i+1}: Guess: ${guess:,.2f} Truth: ${truth:,.2f} Error: ${error:,.2f} SLE: {sle:,.2f} Item: {title}{RESET}")

    def chart(self, title):
        max_error = max(self.errors)
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=2, alpha=0.6)
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel('Valor Real')
        plt.ylabel('Estimación del Modelo')
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.show()

    def report(self):
        average_error = sum(self.errors) / self.size
        rmsle = math.sqrt(sum(self.sles) / self.size)
        hits = sum(1 for color in self.colors if color=="green")
        title = f"{self.title}. Mean error=${average_error:,.2f}. Hits={hits/self.size*100:.1f}%"
        self.chart(title)

    def run(self):
        self.error = 0
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, function, data):
        cls(function, data).run()

# Modelo Base

#### Cuantización

In [ ]:
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

#### Llamada al tokenizador y modelo base

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id


In [ ]:
#arquitectura del modelo base
base_model

#### Pruebas sobre 250 elementos del conjunto de test

In [ ]:
Tester.test(base_model_predict, test)

# Modelo Finetuneado

#### Ajuste con PEFT

In [ ]:
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION) #se coloca modelo base y modelo finetuneado y en revisión colocamos el commit que vamos a usar del finetuneado, en este caso el último
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)


print(f"Impacto en la memoria: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# Arquitectura del modelo
fine_tuned_model

#### Prueba sobre 250 elementos del conjunto de test

In [ ]:
Tester.test(finetuned_model_predict, test)

#### Predicción y prueba alterna:
Esta predicción propone que se consideren los tres tokens con mayor probabilidad y se promedien considerando sus pesos (la probabilidad de cada uno de ellos).

In [ ]:
# Una función de predicción mejorada toma un promedio ponderado de las 3 opciones principales
# Este código sería más complejo si no pudiéramos aprovechar el hecho de que Llama genera 1 token para cualquier número de 3 dígitos

top_K = 3 #toma los siguientes 3 tokens más probables

def improved_model_predict(prompt, device="cuda"):
    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    attention_mask = torch.ones(inputs.shape, device=device)

    with torch.no_grad():
        outputs = fine_tuned_model(inputs, attention_mask=attention_mask)
        next_token_logits = outputs.logits[:, -1, :].to('cpu') #funcion para obtener la lista de logits

    next_token_probs = F.softmax(next_token_logits, dim=-1)#funcion para acceder a la funcion softmax
    top_prob, top_token_id = next_token_probs.topk(top_K) #función para quedarse con los tres tokens de mayor probabilidad
    prices, weights = [], []
    for i in range(top_K): #Para ver los top k tokens
      predicted_token = tokenizer.decode(top_token_id[0][i])
      probability = top_prob[0][i]
      try:
        result = float(predicted_token)
      except ValueError as e:
        result = 0.0
      if result > 0:
        prices.append(result)
        weights.append(probability)
    if not prices:
      return 0.0, 0.0
    total = sum(weights)
    weighted_prices = [price * weight / total for price, weight in zip(prices, weights)] #la nueva predicción se obtiene como el promedio de las tres predicciones más probables, pero con la afectación del peso en cada predicción.
    return sum(weighted_prices).item()

In [ ]:
Tester.test(improved_model_predict, test)